In [26]:
import numpy as np
import pandas as pd

In [27]:
ph_df=pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Photohide/HIDE OUT ACTIVITY.xlsx")

In [ ]:
ph_df.rename(columns={"Unnamed: 0":"Date","Unnamed: 1":"Type of animal","Unnamed: 2":"Quantity","Unnamed: 3":"Time in","Unnamed: 4":"Time out","Unnamed: 5":"Mins. Spent"}, inplace=True)
ph_df.head()

In [ ]:
ph_df.drop(columns=["Unnamed: 6","Unnamed: 7","Unnamed: 8"], inplace=True)
ph_df.head()

In [ ]:
ph_df['Quantity']=pd.to_numeric(ph_df['Quantity'],errors='ignore')

In [ ]:
ph_df.drop(index=[0,1,2], inplace=True)
ph_df.head()

In [32]:
total_appearances=pd.DataFrame(ph_df.groupby('Type of animal')['Date'].count())
total_appearances=total_appearances.sort_values(by=['Date'],ascending=False)
mostcommon=total_appearances.head(20)

In [ ]:
#convert columns to date time objects
ph_df['Mins. Spent']=pd.to_timedelta(ph_df['Mins. Spent'].astype(str),errors='coerce')
ph_df['Date']=pd.to_datetime(ph_df['Date'].astype(str),errors='coerce')
ph_df['S_Date']=ph_df['Date'].astype(str)
ph_df['Time in']=pd.to_datetime(ph_df['Time in'].astype(str),errors='coerce')
ph_df['Time out']=pd.to_datetime(ph_df['Time out'].astype(str),errors='coerce')
ph_df['Quantity']=pd.to_numeric(ph_df['Quantity'],errors='coerce')
ph_df.head()

In [ ]:
animalsmonthly=pd.DataFrame(ph_df.groupby(pd.Grouper(key='Date',freq='M'))['Quantity'].agg([np.sum,np.mean]))
#drop data that is equal to 0
animalsmonthly=animalsmonthly.drop(animalsmonthly.index[animalsmonthly['sum']==0.0])
animalsmonthly.head(80)

In [ ]:
#analysis of total appearances on a weekly basis
animalsWeekly=pd.DataFrame(ph_df.groupby(pd.Grouper(key='Date',freq='W'))['Quantity'].agg([np.sum,np.mean]))
animalsWeekly=animalsWeekly.drop(animalsWeekly.index[animalsWeekly['sum']==0.0])
animalsWeekly.head(20)


In [ ]:
#Average time spent by each animal at the water hide
ph_df['Mins. Spent']=pd.to_timedelta(ph_df['Mins. Spent'],unit='Minutes')
average_time=pd.DataFrame(ph_df.groupby('Type of animal')['Mins. Spent'].agg(np.mean))

average_time.head(30)

In [ ]:
#find total number of animals for each animal hourly
time_in=pd.DataFrame(ph_df.groupby(['Type of animal',(pd.Grouper(key='Time in',freq='H'))])['Quantity'].agg('max'))
time_in.to_excel("time_in.xlsx")
time_in.head(30)
#convert it to file format



In [ ]:

timing=pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Photohide/time_in.xlsx')
timing.head()

In [ ]:
#convert the file into a new df
timing=pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Photohide/time_in.xlsx")
#Find the hours with the highest no. of animals, per animal
likely_time_in=pd.DataFrame(timing.groupby('Type of animal').agg({'No. Of Animals':'max'},'Time in'))
#convert to a file format
likely_time_in.to_excel("likely_time_in.xlsx")
likely_time_in.head(20)


In [87]:
#create a concatenated column to use as a filgter for the joins
timing['joined']=timing['Type of animal']+' '+timing['No. Of Animals'].astype(str)
concat=pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Photohide/likely_time_in.xlsx')
concat['joined']=concat['Type of animal']+' '+concat['No. Of Animals'].astype(str)

In [ ]:
#filter out the values in timing using concat to find the specific peak hours using inner join and merge
mostcommon=pd.merge(timing,concat,on=['joined'],how='inner')
#drop unnecessary columns
mostcommon.drop(columns=['Type of animal_y','No. Of Animals_y','joined','Combo'],inplace=True)
#convert to file format
mostcommon.to_excel("peakhours.xlsx")
mostcommon.head(15)

In [ ]:
total_appearances.to_excel("total_appearances.xlsx")
animalsmonthly.to_excel("animalsmonthly.xlsx")
animalsWeekly.to_excel("animalsWeekly.xlsx")


